In [1]:
%%capture
!git clone -b Trong https://github.com/sitrongtrang/LRL-IR.git
!pip install -r LRL-IR/requirement.txt
!pip install datasets huggingface_hub python-dotenv

In [2]:
import pandas as pd
import os

# Update the .env file to point to Kaggle's working directory
with open(".env", "w") as f:
    f.write("PROJECT_DIR=/kaggle/working/LRL-IR/\n")

# Make sure to replace 'your-dataset-name' with whatever you named your uploaded dataset!
input_csv_path = "/kaggle/input/datasets/thaideptrai3250/nlp-phase1/en_vi_translation.csv"
output_csv_path = "/kaggle/working/LRL-IR/fourThousandRow.csv"

# Load, slice, and save to the working directory
df = pd.read_csv(input_csv_path)
df_new = df[:8000]
df_new.to_csv(output_csv_path, index=False)

df_new.head()

,source,target
0,Please put the dustpan in the broom closet,xin vui lòng đặt người quét rác trong tủ chổi
1,Be quiet for a moment.,im lặng một lát
2,Read this,đọc này
3,Tom persuaded the store manager to give him ba...,tom thuyết phục người quản lý cửa hàng trả lại...
4,Friendship consists of mutual understanding,tình bạn bao gồm sự hiểu biết lẫn nhau


In [3]:
!pip install datasets huggingface_hub
from datasets import load_dataset
from huggingface_hub import login

In [4]:
%%capture
!rm -rf /vncorenlp/models

# Run the training script with updated Kaggle paths
!python LRL-IR/main.py knowledge_distillation en vi distiluse-base-multilingual-cased-v2 distiluse-base-multilingual-cased-v2 /kaggle/working/LRL-IR/fourThousandRow.csv /kaggle/working/LRL-IR/ tf_idf cuda 32 1.0 1e-4 1

# Zip the trained model so you can easily download it from Kaggle's "Output" section
!zip -r /kaggle/working/model_output.zip /kaggle/working/LRL-IR/sentence_transformer_multilingual_tf_idf

In [5]:
print("Done")

Done


## Phase 1 Evaluation — Test student model trên dataset Anh–Việt

Phần này dùng dataset **IWSLT'15 English–Vietnamese trên Kaggle** để đánh giá student model sau Phase 1.

Dataset Kaggle cần add vào notebook:
`/kaggle/input/iwslt15-englishvietnamese`

Metrics:
- `positive_cosine_mean`: cosine giữa câu English và câu Vietnamese đúng cặp.
- `negative_cosine_mean`: cosine giữa câu English và câu Vietnamese sai cặp.
- `margin_mean`: positive cosine - negative cosine.
- `MRR`, `Acc@1`, `Acc@5`, `Acc@10`: đánh giá bilingual sentence retrieval.


In [1]:
# =========================
# PHASE 1 TEST CONFIG
# =========================
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Model teacher dùng để encode source English
TEACHER_MODEL_NAME = "distiluse-base-multilingual-cased-v2"

# Student model sau Phase 1. Nếu bạn test model từ Kaggle input thì thay path này.
STUDENT_MODEL_PATH = "/kaggle/input/datasets/thaideptrai3250/phase1-dataset/Phase1_2000/kaggle/working/LRL-IR/sentence_transformer_multilingual_tf_idf"

# Dataset Kaggle bạn gửi:
# https://www.kaggle.com/datasets/tuannguyenvananh/iwslt15-englishvietnamese
# Sau khi Add Data vào notebook, Kaggle thường mount ở path này.
KAGGLE_IWSLT_ROOT = "/kaggle/input/datasets/tuannguyenvananh/iwslt15-englishvietnamese"

# File eval CSV chuẩn hóa sẽ được tạo từ dataset Kaggle local.
EVAL_CSV = "/kaggle/working/iwslt15_en_vi_test.csv"

# Nếu muốn test nhanh, để 1000 hoặc 2000. Nếu muốn dùng toàn bộ, đặt None.
MAX_EVAL_SAMPLES = 1000

BATCH_SIZE = 64
NUM_NEGATIVES = 5

print("Device:", DEVICE)
print("Student model path:", STUDENT_MODEL_PATH)
print("Kaggle IWSLT root:", KAGGLE_IWSLT_ROOT)


Device: cuda
Student model path: /kaggle/input/datasets/thaideptrai3250/phase1-dataset/Phase1_2000/kaggle/working/LRL-IR/sentence_transformer_multilingual_tf_idf
Kaggle IWSLT root: /kaggle/input/datasets/tuannguyenvananh/iwslt15-englishvietnamese


In [3]:
from pathlib import Path
import pandas as pd
import re

def clean_text(text):
    text = str(text)
    text = text.replace("\ufeff", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def find_file_by_name(filename, root="/kaggle/input/datasets/tuannguyenvananh/iwslt15-englishvietnamese/IWSLT'15 en-vi"):
    matches = list(Path(root).rglob(filename))
    if len(matches) == 0:
        raise FileNotFoundError(f"Không tìm thấy file: {filename} trong {root}")
    return matches[0]

def load_parallel_txt(en_path, vi_path, max_samples=None):
    en_path = Path(en_path)
    vi_path = Path(vi_path)

    with open(en_path, "r", encoding="utf-8", errors="ignore") as f:
        en_lines = f.readlines()

    with open(vi_path, "r", encoding="utf-8", errors="ignore") as f:
        vi_lines = f.readlines()

    print("English lines:", len(en_lines))
    print("Vietnamese lines:", len(vi_lines))

    if len(en_lines) != len(vi_lines):
        print("Warning: số dòng EN và VI không bằng nhau. Sẽ lấy theo số dòng nhỏ hơn.")

    rows = []
    for en, vi in zip(en_lines, vi_lines):
        en = clean_text(en)
        vi = clean_text(vi)

        if en == "" or vi == "":
            continue

        rows.append({
            "source": en,
            "target": vi
        })

    df = pd.DataFrame(rows)

    if max_samples is not None:
        df = df.head(max_samples).reset_index(drop=True)

    return df
    
en_file = find_file_by_name("tst2012.en.txt")
vi_file = find_file_by_name("tst2012.vi.txt")

print("EN file:", en_file)
print("VI file:", vi_file)

test_df = load_parallel_txt(
    en_file,
    vi_file,
    max_samples=None
)

print(test_df.shape)
test_df.head()

EN file: /kaggle/input/datasets/tuannguyenvananh/iwslt15-englishvietnamese/IWSLT'15 en-vi/tst2012.en.txt
VI file: /kaggle/input/datasets/tuannguyenvananh/iwslt15-englishvietnamese/IWSLT'15 en-vi/tst2012.vi.txt
English lines: 1553
Vietnamese lines: 1553
(1553, 2)


,source,target
0,How can I speak in 10 minutes about the bonds ...,Làm sao tôi có thể trình bày trong 10 phút về ...
1,This is not a finished story .,Câu chuyện này chưa kết thúc .
2,It is a jigsaw puzzle still being put together .,Nó là một trò chơi ghép hình vẫn đang được xếp .
3,Let me tell you about some of the pieces .,Hãy để tôi kể cho các bạn về vài mảnh ghép nhé .
4,Imagine the first piece : a man burning his li...,Hãy tưởng tượng mảnh đầu tiên : một người đàn ...


In [4]:
EVAL_CSV = "/kaggle/working/iwslt15_en_vi_tst2012.csv"
test_df.to_csv(EVAL_CSV, index=False)

print("Saved:", EVAL_CSV)
english_sentences = test_df["source"].tolist()
vietnamese_sentences = test_df["target"].tolist()

print("Number of eval pairs:", len(english_sentences))
print("Example EN:", english_sentences[0])
print("Example VI:", vietnamese_sentences[0])

Saved: /kaggle/working/iwslt15_en_vi_tst2012.csv
Number of eval pairs: 1553
Example EN: How can I speak in 10 minutes about the bonds of women over three generations , about how the astonishing strength of those bonds took hold in the life of a four-year-old girl huddled with her young sister , her mother and her grandmother for five days and nights in a small boat in the China Sea more than 30 years ago , bonds that took hold in the life of that small girl and never let go -- that small girl now living in San Francisco and speaking to you today ?
Example VI: Làm sao tôi có thể trình bày trong 10 phút về sợi dây liên kết những người phụ nữ qua ba thế hệ , về việc làm thế nào những sợi dây mạnh mẽ đáng kinh ngạc ấy đã níu chặt lấy cuộc sống của một cô bé bốn tuổi co quắp với đứa em gái nhỏ của cô bé , với mẹ và bà trong suốt năm ngày đêm trên con thuyền nhỏ lênh đênh trên Biển Đông hơn 30 năm trước , những sợi dây liên kết đã níu lấy cuộc đời cô bé ấy và không bao giờ rời đi -- cô bé ấy

In [5]:
# =========================
# EVALUATE: COSINE + BILINGUAL RETRIEVAL
# =========================

def compute_negative_scores(source_emb, target_emb, num_negatives=5, seed=42):
    """
    source_emb, target_emb đã được normalize.
    Tạo negative pair bằng cách shuffle target.
    """
    rng = np.random.default_rng(seed)
    n = len(source_emb)
    all_scores = []

    for _ in range(num_negatives):
        perm = rng.permutation(n)

        if n > 1:
            same_idx = np.where(perm == np.arange(n))[0]
            for idx in same_idx:
                swap_idx = (idx + 1) % n
                perm[idx], perm[swap_idx] = perm[swap_idx], perm[idx]

        scores = np.sum(source_emb * target_emb[perm], axis=1)
        all_scores.append(scores)

    return np.concatenate(all_scores)


def compute_retrieval_metrics(source_emb, target_emb, batch_size=256, ks=(1, 5, 10)):
    """
    English source embedding làm query.
    Vietnamese target embedding làm candidates.
    Correct candidate của query i là target i.
    """
    n = source_emb.shape[0]
    ranks = []

    target_T = target_emb.T

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        sims = source_emb[start:end] @ target_T

        for local_i, row in enumerate(sims):
            global_i = start + local_i
            correct_score = row[global_i]

            # rank 1 = tốt nhất
            rank = int(np.sum(row > correct_score) + 1)
            ranks.append(rank)

    ranks = np.array(ranks)

    result = {
        "MRR": float(np.mean(1.0 / ranks)),
        "mean_rank": float(np.mean(ranks)),
        "median_rank": float(np.median(ranks)),
    }

    for k in ks:
        result[f"Acc@{k}"] = float(np.mean(ranks <= k))

    return result, ranks


def evaluate_phase1_student(
    teacher_model_name,
    student_model_path,
    eval_df,
    batch_size=64,
    num_negatives=5,
    device="cuda",
):
    print("Loading teacher:", teacher_model_name)
    teacher = SentenceTransformer(teacher_model_name, device=device)

    print("Loading student:", student_model_path)
    student = SentenceTransformer(student_model_path, device=device)

    print("\nTeacher architecture:")
    print(teacher)

    print("\nStudent architecture:")
    print(student)
    print("Student model_type:", student[0].auto_model.config.model_type)
    print("Student architecture:", student[0].auto_model.__class__.__name__)

    english_sentences = eval_df["source"].tolist()
    vietnamese_sentences = eval_df["target"].tolist()

    print("\nEncoding English with teacher...")
    source_emb = teacher.encode(
        english_sentences,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    print("\nEncoding Vietnamese with student...")
    target_emb = student.encode(
        vietnamese_sentences,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    if source_emb.shape[1] != target_emb.shape[1]:
        raise ValueError(
            f"Embedding dim mismatch: teacher={source_emb.shape[1]}, student={target_emb.shape[1]}. "
            "Hãy kiểm tra teacher/student có cùng output dimension không."
        )

    positive_scores = np.sum(source_emb * target_emb, axis=1)
    negative_scores = compute_negative_scores(
        source_emb,
        target_emb,
        num_negatives=num_negatives,
        seed=SEED,
    )

    retrieval_metrics, ranks = compute_retrieval_metrics(source_emb, target_emb)

    metrics = {
        "model_path": student_model_path,
        "num_eval_pairs": len(eval_df),

        "positive_cosine_mean": float(np.mean(positive_scores)),
        "positive_cosine_std": float(np.std(positive_scores)),
        "negative_cosine_mean": float(np.mean(negative_scores)),
        "negative_cosine_std": float(np.std(negative_scores)),
        "margin_mean": float(np.mean(positive_scores) - np.mean(negative_scores)),

        **retrieval_metrics,
    }

    results_df = pd.DataFrame([metrics])
    return results_df, positive_scores, negative_scores, ranks


phase1_results_df, positive_scores, negative_scores, ranks = evaluate_phase1_student(
    teacher_model_name=TEACHER_MODEL_NAME,
    student_model_path=STUDENT_MODEL_PATH,
    eval_df=test_df,
    batch_size=BATCH_SIZE,
    num_negatives=NUM_NEGATIVES,
    device=DEVICE,
)

display(phase1_results_df)


Loading teacher: distiluse-base-multilingual-cased-v2


modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...


Teacher architecture:
SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False, 'architecture': 'DistilBertModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 512, 'bias': True, 'activation_function': 'torch.nn.modules.activation.Tanh'})
)

Student architecture:
SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'DistilBertModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_p

Batches:   0%|          | 0/25 [00:00<?, ?it/s]


Encoding Vietnamese with student...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

,model_path,num_eval_pairs,positive_cosine_mean,positive_cosine_std,negative_cosine_mean,negative_cosine_std,margin_mean,MRR,mean_rank,median_rank,Acc@1,Acc@5,Acc@10
0,/kaggle/input/datasets/thaideptrai3250/phase1-...,1553,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0


In [6]:
import numpy as np

def check_embedding(name, emb):
    print("=" * 80)
    print(name)
    print("shape:", emb.shape)
    print("dtype:", emb.dtype)
    print("NaN count:", np.isnan(emb).sum())
    print("Inf count:", np.isinf(emb).sum())

    norms = np.linalg.norm(emb, axis=1)
    print("Norm NaN count:", np.isnan(norms).sum())
    print("Norm zero count:", np.sum(norms == 0))
    print("Norm min:", np.nanmin(norms))
    print("Norm max:", np.nanmax(norms))
    print("Norm mean:", np.nanmean(norms))

check_embedding("Teacher embeddings", teacher_emb)
check_embedding("Student embeddings", student_emb)

NameError: name 'teacher_emb' is not defined

In [ ]:
# =========================
# SAVE RESULT
# =========================
result_csv = "/kaggle/working/phase1_student_test_results.csv"
phase1_results_df.to_csv(result_csv, index=False)

detail_csv = "/kaggle/working/phase1_student_test_details.csv"
detail_df = eval_df.copy()
detail_df["positive_cosine"] = positive_scores
detail_df["rank"] = ranks
detail_df.to_csv(detail_csv, index=False)

print("Saved summary:", result_csv)
print("Saved details:", detail_csv)

display(detail_df.sort_values("rank", ascending=False).head(10))
